### Task 2 - Feature Engineering Challenge Using NYC Bike Share Data
This notebook uses the NYC Bike Share dataset to create new features from date and time information, apply feature engineering techniques, and compare Logistic Regression model performance before and after feature engineering. The goal is to measure how engineered features improve prediction accuracy.

#### Import Libraries

In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

### Load Dataset

In [12]:
# Read dataset

df = pd.read_csv("NYC-BikeShare-2015-2017.csv")
print("Shape:", df.shape)
df.head()

Shape: (735502, 17)


,Unnamed: 0,Trip Duration,Start Time,Stop Time,Start Station ID,Start Station Name,Start Station Latitude,Start Station Longitude,End Station ID,End Station Name,End Station Latitude,End Station Longitude,Bike ID,User Type,Birth Year,Gender,Trip_Duration_in_min
0,0,376,2015-10-01 00:16:26,2015-10-01 00:22:42,3212,Christ Hospital,40.734786,-74.050444,3207,Oakland Ave,40.737604,-74.052478,24470,Subscriber,1960.0,1,6
1,1,739,2015-10-01 00:27:12,2015-10-01 00:39:32,3207,Oakland Ave,40.737604,-74.052478,3212,Christ Hospital,40.734786,-74.050444,24481,Subscriber,1960.0,1,12
2,2,2714,2015-10-01 00:32:46,2015-10-01 01:18:01,3193,Lincoln Park,40.724605,-74.078406,3193,Lincoln Park,40.724605,-74.078406,24628,Subscriber,1983.0,1,45
3,3,275,2015-10-01 00:34:31,2015-10-01 00:39:06,3199,Newport Pkwy,40.728745,-74.032108,3187,Warren St,40.721124,-74.038051,24613,Subscriber,1975.0,1,5
4,4,561,2015-10-01 00:40:12,2015-10-01 00:49:33,3183,Exchange Place,40.716247,-74.033459,3192,Liberty Light Rail,40.711242,-74.055701,24668,Customer,1984.0,0,9


In [13]:
# Check data types
print(df.dtypes)

Unnamed: 0                   int64
Trip Duration                int64
Start Time                     str
Stop Time                      str
Start Station ID             int64
Start Station Name             str
Start Station Latitude     float64
Start Station Longitude    float64
End Station ID               int64
End Station Name               str
End Station Latitude       float64
End Station Longitude      float64
Bike ID                      int64
User Type                      str
Birth Year                 float64
Gender                       int64
Trip_Duration_in_min         int64
dtype: object


In [16]:
# Check missing values
print(df.isnull().sum())

# Check target distribution
print(df["User Type"].value_counts())

Unnamed: 0                 0
Trip Duration              0
Start Time                 0
Stop Time                  0
Start Station ID           0
Start Station Name         0
Start Station Latitude     0
Start Station Longitude    0
End Station ID             0
End Station Name           0
End Station Latitude       0
End Station Longitude      0
Bike ID                    0
User Type                  0
Birth Year                 0
Gender                     0
Trip_Duration_in_min       0
target                     0
dtype: int64
User Type
Subscriber    688140
Customer       47362
Name: count, dtype: int64


### Create Target Variable

In [17]:
### Create Target Variable
# Subscriber = 1
# Customer = 0

df["target"] = (
    df["User Type"] == "Subscriber"
).astype(int)

print(df["target"].value_counts())

target
1    688140
0     47362
Name: count, dtype: int64


### Create Feature Engineered Columns

In [22]:
# Convert to datetime

df["Start Time"] = pd.to_datetime(df["Start Time"])
# Feature 1
# Extract hour

df["start_hour"] = df["Start Time"].dt.hour

# Feature 2
# Extract day name

df["day_of_week"] = df["Start Time"].dt.day_name()

# Feature 3
# Weekend flag

df["is_weekend"] = (
    df["Start Time"].dt.dayofweek >= 5
).astype(int)

# Create rider age
df["age"] = 2017 - df["Birth Year"]

# Feature 4
# Interaction feature
# Combine age and trip duration into one feature
# This helps the model learn patterns that depend on both values together
df["age_x_duration"] = (
    df["age"] * df["Trip_Duration_in_min"]
)

# Feature 5
# Interaction feature

df["duration_per_age"] = (
    df["Trip_Duration_in_min"] / (df["age"] + 1)
)

# Feature 6
# Log transform

df["log_duration"] = np.log1p(
    df["Trip_Duration_in_min"]
)

# Feature 7
# Age groups

df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 25, 40, 60, 100],
    labels=[
        "Young",
        "Adult",
        "Middle",
        "Senior"
    ]
)

df.head()

,Unnamed: 0,Trip Duration,Start Time,Stop Time,Start Station ID,Start Station Name,Start Station Latitude,Start Station Longitude,End Station ID,End Station Name,...,Trip_Duration_in_min,target,start_hour,day_of_week,is_weekend,age,age_x_duration,duration_per_age,log_duration,age_group
0,0,376,2015-10-01 00:16:26,2015-10-01 00:22:42,3212,Christ Hospital,40.734786,-74.050444,3207,Oakland Ave,...,6,1,0,Thursday,0,57.0,342.0,0.103448,1.945910,Middle
1,1,739,2015-10-01 00:27:12,2015-10-01 00:39:32,3207,Oakland Ave,40.737604,-74.052478,3212,Christ Hospital,...,12,1,0,Thursday,0,57.0,684.0,0.206897,2.564949,Middle
2,2,2714,2015-10-01 00:32:46,2015-10-01 01:18:01,3193,Lincoln Park,40.724605,-74.078406,3193,Lincoln Park,...,45,1,0,Thursday,0,34.0,1530.0,1.285714,3.828641,Adult
3,3,275,2015-10-01 00:34:31,2015-10-01 00:39:06,3199,Newport Pkwy,40.728745,-74.032108,3187,Warren St,...,5,1,0,Thursday,0,42.0,210.0,0.116279,1.791759,Middle
4,4,561,2015-10-01 00:40:12,2015-10-01 00:49:33,3183,Exchange Place,40.716247,-74.033459,3192,Liberty Light Rail,...,9,0,0,Thursday,0,33.0,297.0,0.264706,2.302585,Adult


### Baseline Model Using Original Features

In [23]:
baseline_features = [
    "Gender",
    "Birth Year",
    "Trip_Duration_in_min"
]

X = df[baseline_features]

y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

### Train Baseline Model

In [24]:
baseline_pipeline = Pipeline(
    [
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(max_iter=1000)
        )
    ]
)

baseline_pipeline.fit(
    X_train,
    y_train
)

baseline_predictions = baseline_pipeline.predict(
    X_test
)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

print("Baseline Accuracy:", baseline_accuracy)

Baseline Accuracy: 0.9833379786677181


### Model Using Engineered Features

In [26]:
engineered_features = [
    "Gender",
    "Birth Year",
    "Trip_Duration_in_min",

    "start_hour",
    "day_of_week",
    "is_weekend",

    "age_x_duration",
    "duration_per_age",

    "log_duration",
    "age_group"
]

X = df[engineered_features]

y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

### Train Feature Engineered Model

In [28]:
numeric_columns = [
    "Gender",
    "Birth Year",
    "Trip_Duration_in_min",
    "start_hour",
    "is_weekend",
    "age_x_duration",
    "duration_per_age",
    "log_duration"
]

categorical_columns = [
    "day_of_week",
    "age_group"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    (
                        "imputer",
                        SimpleImputer(strategy="median")
                    ),
                    (
                        "scaler",
                        StandardScaler()
                    )
                ]
            ),
            numeric_columns
        ),
        (
            "cat",
            Pipeline(
                [
                    (
                        "imputer",
                        SimpleImputer(strategy="most_frequent")
                    ),
                    (
                        "encoder",
                        OneHotEncoder(handle_unknown="ignore")
                    )
                ]
            ),
            categorical_columns
        )
    ]
)

engineered_pipeline = Pipeline(
    [
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(max_iter=1000)
        )
    ]
)

engineered_pipeline.fit(
    X_train,
    y_train
)

engineered_predictions = engineered_pipeline.predict(
    X_test
)

engineered_accuracy = accuracy_score(
    y_test,
    engineered_predictions
)

print("Feature Engineered Accuracy:", engineered_accuracy)

Feature Engineered Accuracy: 0.9874100108089


### Compare Results

In [30]:
accuracy_delta = (
    engineered_accuracy - baseline_accuracy
)

print("Baseline Accuracy :", round(baseline_accuracy, 4))

print("Engineered Accuracy :", round(engineered_accuracy, 4))

print("Accuracy Improvement :", round(accuracy_delta, 4))

Baseline Accuracy : 0.9833
Engineered Accuracy : 0.9874
Accuracy Improvement : 0.0041


### Feature Engineering Notes :

In [32]:
# start_hour
# Riders behave differently during morning, afternoon and evening.

# day_of_week
# Riding patterns are different on weekdays and weekends.

# is_weekend
# Weekend trips often have different purposes.

# age_x_duration
# Combines rider age and trip duration.

# duration_per_age
# Measures trip duration relative to rider age.

# log_duration
# Reduces skewness from extremely long trips.

# age_group
# Converts exact ages into meaningful categories.